In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import glorot, uniform
from torch_geometric.utils import softmax
import copy
import random
from collections import defaultdict
from torch_geometric.utils import negative_sampling
import math
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
import anndata as ad
ad.settings.allow_write_nullable_strings = True
from torch_geometric.nn import GATv2Conv

#### AE模型定义

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AutoEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims=(512,),
        latent_dim: int = 256,
        dropout: float = 0.0,
        output_activation: str | None = None,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_activation = output_activation

        enc_dims = [input_dim] + list(hidden_dims) + [latent_dim]
        enc_layers = []
        for i in range(len(enc_dims) - 1):
            enc_layers.append(nn.Linear(enc_dims[i], enc_dims[i + 1]))
            if i != len(enc_dims) - 2:
                enc_layers.append(nn.ReLU())
                if dropout > 0:
                    enc_layers.append(nn.Dropout(dropout))
        self.encoder = nn.Sequential(*enc_layers)

        dec_dims = [latent_dim] + list(hidden_dims[::-1]) + [input_dim]
        dec_layers = []
        for i in range(len(dec_dims) - 1):
            dec_layers.append(nn.Linear(dec_dims[i], dec_dims[i + 1]))
            if i != len(dec_dims) - 2:
                dec_layers.append(nn.ReLU())
                if dropout > 0:
                    dec_layers.append(nn.Dropout(dropout))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x_hat = self.decoder(z)

        if self.output_activation == "relu":
            x_hat = F.relu(x_hat)
        elif self.output_activation == "sigmoid":
            x_hat = torch.sigmoid(x_hat)
        elif self.output_activation is None:
            pass
        else:
            raise ValueError(f"Unsupported output_activation: {self.output_activation}")

        return x_hat

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    @torch.no_grad()
    def get_embedding(self, x: torch.Tensor, batch_size: int = 1024) -> torch.Tensor:
        self.eval()
        outs = []
        for start in range(0, x.shape[0], batch_size):
            batch = x[start:start + batch_size]
            z = self.encode(batch)
            outs.append(z.cpu())
        return torch.cat(outs, dim=0)

#### AE模型训练

In [8]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split


def train_autoencoder(
    model,
    x: torch.Tensor,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 1024,
    max_epochs: int = 500,
    val_ratio: float = 0.1,
    patience: int = 30,
    min_delta: float = 1e-5,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    verbose: bool = True,
):
    """
    Train one autoencoder on matrix x.
    x shape: [n_samples, n_features]
    """
    model = model.to(device)
    x = x.to(torch.float32)

    dataset = TensorDataset(x)
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val

    train_set, val_set = random_split(
        dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(0)
    )

    train_loader = DataLoader(
        train_set,
        batch_size=min(batch_size, len(train_set)),
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=min(batch_size, len(val_set)),
        shuffle=False,
        drop_last=False,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    wait = 0

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(max_epochs):
        model.train()
        train_losses = []

        for (batch_x,) in train_loader:
            batch_x = batch_x.to(device)

            x_hat, z = model(batch_x)
            loss = criterion(x_hat, batch_x)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(device)
                x_hat, z = model(batch_x)
                loss = criterion(x_hat, batch_x)
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if verbose and (epoch % 20 == 0 or epoch == max_epochs - 1):
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_loss:.6f} | "
                f"val_loss={val_loss:.6f}"
            )

        if val_loss < best_val - min_delta:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch:03d}, best_val={best_val:.6f}")
                break

    model.load_state_dict(best_state)
    return model, history

In [9]:
def fit_gene_cell_autoencoders(
    gene_cell: np.ndarray,
    latent_dim: int = 256,
    hidden_dims=(512,),
    dropout: float = 0.0,
    output_activation=None,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 1024,
    max_epochs: int = 500,
    val_ratio: float = 0.1,
    patience: int = 30,
    min_delta: float = 1e-5,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    verbose: bool = True,
):
    """
    gene_cell shape: [n_genes, n_cells]

    Returns:
        gene_embedding: [n_genes, latent_dim]
        cell_embedding: [n_cells, latent_dim]
    """
    gene_cell = gene_cell.astype(np.float32)

    # gene AE input: gene x cell
    gene_x = torch.tensor(gene_cell, dtype=torch.float32)

    # cell AE input: cell x gene
    cell_x = torch.tensor(gene_cell.T, dtype=torch.float32)

    gene_ae = AutoEncoder(
        input_dim=gene_x.shape[1],
        hidden_dims=hidden_dims,
        latent_dim=latent_dim,
        dropout=dropout,
        output_activation=output_activation,
    )

    cell_ae = AutoEncoder(
        input_dim=cell_x.shape[1],
        hidden_dims=hidden_dims,
        latent_dim=latent_dim,
        dropout=dropout,
        output_activation=output_activation,
    )

    if verbose:
        print("Training gene AE...")
    gene_ae, gene_history = train_autoencoder(
        model=gene_ae,
        x=gene_x,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        max_epochs=max_epochs,
        val_ratio=val_ratio,
        patience=patience,
        min_delta=min_delta,
        device=device,
        verbose=verbose,
    )

    if verbose:
        print("Training cell AE...")
    cell_ae, cell_history = train_autoencoder(
        model=cell_ae,
        x=cell_x,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        max_epochs=max_epochs,
        val_ratio=val_ratio,
        patience=patience,
        min_delta=min_delta,
        device=device,
        verbose=verbose,
    )

    gene_embedding = gene_ae.get_embedding(
        gene_x.to(device),
        batch_size=batch_size
    ).numpy()

    cell_embedding = cell_ae.get_embedding(
        cell_x.to(device),
        batch_size=batch_size
    ).numpy()

    return {
        "gene_ae": gene_ae, # 训练好的 gene autoencoder 模型对象
        "cell_ae": cell_ae, # 训练好的 cell autoencoder 模型对象
        "gene_embedding": gene_embedding, # [n_genes, latent_dim]
        "cell_embedding": cell_embedding, # [n_cells, latent_dim]
        "gene_history": gene_history,   # training history for gene AE
        "cell_history": cell_history,   # training history for cell AE
    }

#### 训练过程

In [10]:
adata_sub = sc.read_h5ad("/home/user/jiayuran/Omics/Adenomyosis/Data/ProcessedGSE218044/sc_adenomyosis_hvg_1000cells_by_major_celltype.h5ad")
# =========================
# 1. 只保留高变基因
# =========================
hvg_mask = adata_sub.var["var.features"].to_numpy(dtype=bool)
print("HVG数量:", hvg_mask.sum())

adata_sub = adata_sub[:, hvg_mask].copy()
print("HVG后维度:", adata_sub.shape)
print(adata_sub)

HVG数量: 2000
HVG后维度: (1000, 2000)
AnnData object with n_obs × n_vars = 1000 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'group', 'percent.mt', 'RNA_snn_res.0.8', 'cluster', 'major_celltype', 'ident'
    var: 'var.features'
    obsm: 'HARMONY', 'X_pca', 'X_umap'
    layers: 'logcounts'


In [11]:
# 取表达矩阵
X = adata_sub.X

# sparse -> dense
if sp.issparse(X):
    X = X.toarray()
else:
    X = np.asarray(X)

# 转成 gene x cell
gene_cell = X.T.astype(np.float32)

print("gene_cell shape:", gene_cell.shape)   # [n_genes, n_cells]

gene_cell shape: (2000, 1000)


In [12]:
result = fit_gene_cell_autoencoders(
    gene_cell=gene_cell,
    latent_dim=256,
    hidden_dims=(512,),
    dropout=0.0,
    output_activation=None,
    lr=1e-3,
    weight_decay=0.0,
    batch_size=1024,
    max_epochs=100,
    val_ratio=0.1,
    patience=30,
)

gene_embedding = result["gene_embedding"]
cell_embedding = result["cell_embedding"]

print(gene_embedding.shape)  # [n_genes, 256]
print(cell_embedding.shape)  # [n_cells, 256]

Training gene AE...
Epoch 000 | train_loss=26.722664 | val_loss=20.820869
Epoch 020 | train_loss=4.271678 | val_loss=8.687460
Epoch 040 | train_loss=2.210340 | val_loss=6.713426
Epoch 060 | train_loss=1.574752 | val_loss=6.017435
Epoch 080 | train_loss=1.289477 | val_loss=5.718660
Epoch 099 | train_loss=1.157913 | val_loss=5.706207
Training cell AE...
Epoch 000 | train_loss=19.258245 | val_loss=83.803299
Epoch 020 | train_loss=7.011443 | val_loss=59.016895
Epoch 040 | train_loss=3.830598 | val_loss=7.323197
Epoch 060 | train_loss=2.594489 | val_loss=4.065239
Epoch 080 | train_loss=2.022104 | val_loss=3.360364
Epoch 099 | train_loss=1.666251 | val_loss=3.129552
(2000, 256)
(1000, 256)


In [13]:
gene_names = adata_sub.var_names.to_numpy()
cell_names = adata_sub.obs_names.to_numpy()

print(len(gene_names), len(cell_names))
print(gene_embedding.shape, cell_embedding.shape)

2000 1000
(2000, 256) (1000, 256)


#### 建图

In [14]:
#### 构建基因-细胞图，只要有表达就连边
def build_gene_cell_graph(gene_cell: np.ndarray):
    """
    gene_cell: [n_genes, n_cells]

    Returns a lightweight graph dict.
    """
    gene_cell = np.asarray(gene_cell)
    n_genes, n_cells = gene_cell.shape

    g_idx, c_idx = np.nonzero(gene_cell)

    # adjacency dictionaries
    gene_to_cells = defaultdict(list)
    cell_to_genes = defaultdict(list)

    for g, c in zip(g_idx, c_idx):
        gene_to_cells[int(g)].append(int(c))
        cell_to_genes[int(c)].append(int(g))

    graph = {
        "n_genes": n_genes,
        "n_cells": n_cells,
        "gene_to_cells": gene_to_cells,
        "cell_to_genes": cell_to_genes,
    }
    return graph

In [15]:
#### 子图采样函数
def sample_bipartite_subgraph(
    graph,
    seed_genes,
    sample_depth: int = 2,
    sample_width_gene_to_cell: int = 64,
    sample_width_cell_to_gene: int = 64,
):
    """
    Sample a bipartite gene-cell subgraph from seed genes.

    Returns:
        sub_genes: sorted list of global gene ids
        sub_cells: sorted list of global cell ids
        edges_gc: list of (local_gene_idx, local_cell_idx)
    """
    gene_to_cells = graph["gene_to_cells"]
    cell_to_genes = graph["cell_to_genes"]

    sub_genes = set(int(g) for g in seed_genes)
    sub_cells = set()

    frontier_genes = set(sub_genes)
    frontier_cells = set()

    for _ in range(sample_depth):
        # expand gene -> cell
        new_cells = set()
        for g in frontier_genes:
            neigh_cells = gene_to_cells.get(g, [])
            if len(neigh_cells) > sample_width_gene_to_cell:
                neigh_cells = random.sample(neigh_cells, sample_width_gene_to_cell)
            new_cells.update(neigh_cells)

        frontier_cells = new_cells - sub_cells
        sub_cells.update(new_cells)

        # expand cell -> gene
        new_genes = set()
        for c in frontier_cells:
            neigh_genes = cell_to_genes.get(c, [])
            if len(neigh_genes) > sample_width_cell_to_gene:
                neigh_genes = random.sample(neigh_genes, sample_width_cell_to_gene)
            new_genes.update(neigh_genes)

        frontier_genes = new_genes - sub_genes
        sub_genes.update(new_genes)

    sub_genes = sorted(sub_genes)
    sub_cells = sorted(sub_cells)

    gene_map = {g: i for i, g in enumerate(sub_genes)}
    cell_map = {c: i for i, c in enumerate(sub_cells)}

    edges_gc = []
    for g in sub_genes:
        for c in gene_to_cells.get(g, []):
            if c in cell_map:
                edges_gc.append((gene_map[g], cell_map[c]))

    return sub_genes, sub_cells, edges_gc


def make_subgraph_jobs(
    graph,
    n_batch: int = 50,
    seed_gene_batch_size: int = 128,
    sample_depth: int = 2,
    sample_width_gene_to_cell: int = 64,
    sample_width_cell_to_gene: int = 64,
    seed: int = 0,
):
    random.seed(seed)
    np.random.seed(seed)

    n_genes = graph["n_genes"]
    all_genes = np.arange(n_genes)

    jobs = []
    for _ in range(n_batch):
        seed_genes = np.random.choice(
            all_genes,
            size=min(seed_gene_batch_size, n_genes),
            replace=False
        )
        sub_genes, sub_cells, edges_gc = sample_bipartite_subgraph(
            graph=graph,
            seed_genes=seed_genes,
            sample_depth=sample_depth,
            sample_width_gene_to_cell=sample_width_gene_to_cell,
            sample_width_cell_to_gene=sample_width_cell_to_gene,
        )
        jobs.append({
            "sub_genes": sub_genes,
            "sub_cells": sub_cells,
            "edges_gc": edges_gc,
        })
    return jobs

def prepare_subgraph_tensors(
    gene_cell: np.ndarray,
    job: dict,
    feature_mode: str = "ae",   # "ae" or "raw"
    gene_embedding: np.ndarray = None,
    cell_embedding: np.ndarray = None,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    """
    Convert one sampled subgraph into tensors for GNN/HGT.
    """

    sub_genes = job["sub_genes"]
    sub_cells = job["sub_cells"]
    edges_gc = job["edges_gc"]

    # submatrix target
    adj_sub = gene_cell[np.ix_(sub_genes, sub_cells)].astype(np.float32)

    # node features
    if feature_mode == "ae":
        assert gene_embedding is not None and cell_embedding is not None
        gene_feat = gene_embedding[sub_genes].astype(np.float32)
        cell_feat = cell_embedding[sub_cells].astype(np.float32)

    elif feature_mode == "raw":
        # gene feature = rows of gene_cell -> [sub_genes, all_cells] or [sub_genes, n_cells]
        # cell feature = rows of gene_cell.T -> [sub_cells, all_genes]
        gene_feat = gene_cell[sub_genes, :].astype(np.float32)
        cell_feat = gene_cell[:, sub_cells].T.astype(np.float32)
    else:
        raise ValueError("feature_mode must be 'ae' or 'raw'")

    # combine nodes
    n_g = len(sub_genes)
    n_c = len(sub_cells)

    if feature_mode == "ae":
        node_feature = np.concatenate([gene_feat, cell_feat], axis=0)   # [n_g+n_c, d]
    else:
        # raw mode: dimensions differ, return separately
        node_feature = [gene_feat, cell_feat]

    node_type = np.concatenate([
        np.zeros(n_g, dtype=np.int64),
        np.ones(n_c, dtype=np.int64)
    ])

    # build bidirectional edges
    # local gene ids: 0 .. n_g-1
    # local cell ids: n_g .. n_g+n_c-1 in unified graph
    src_gc = np.array([g for g, c in edges_gc], dtype=np.int64)
    dst_gc = np.array([c + n_g for g, c in edges_gc], dtype=np.int64)

    src_cg = dst_gc.copy()
    dst_cg = src_gc.copy()

    edge_index = np.vstack([
        np.concatenate([src_gc, src_cg]),
        np.concatenate([dst_gc, dst_cg])
    ])

    # edge type: 0 = gene->cell, 1 = cell->gene
    edge_type = np.concatenate([
        np.zeros(len(src_gc), dtype=np.int64),
        np.ones(len(src_cg), dtype=np.int64)
    ])

    # placeholder edge_time, same as original implementation idea
    edge_time = np.zeros(edge_index.shape[1], dtype=np.int64)

    # tensors
    if feature_mode == "ae":
        node_feature = torch.tensor(node_feature, dtype=torch.float32, device=device)
    else:
        node_feature = [
            torch.tensor(node_feature[0], dtype=torch.float32, device=device),
            torch.tensor(node_feature[1], dtype=torch.float32, device=device),
        ]

    node_type = torch.tensor(node_type, dtype=torch.long, device=device)
    edge_index = torch.tensor(edge_index, dtype=torch.long, device=device)
    edge_type = torch.tensor(edge_type, dtype=torch.long, device=device)
    edge_time = torch.tensor(edge_time, dtype=torch.long, device=device)
    adj_sub = torch.tensor(adj_sub, dtype=torch.float32, device=device)

    return {
        "node_feature": node_feature,
        "node_type": node_type,
        "edge_index": edge_index,
        "edge_type": edge_type,
        "edge_time": edge_time,
        "adj_sub": adj_sub,
        "n_g": n_g,
        "n_c": n_c,
        "sub_genes": sub_genes,
        "sub_cells": sub_cells,
    }

In [18]:
#### KNN构图
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors

def build_knn_edge_index(emb, k=10, metric="cosine", include_self=False, mutual=False):
    """
    emb: numpy array or torch tensor, shape [N, d]
    return: edge_index [2, E]
    """
    if isinstance(emb, torch.Tensor):
        emb = emb.detach().cpu().numpy()

    n = emb.shape[0]
    n_neighbors = k + 1 if include_self else k

    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric=metric)
    nbrs.fit(emb)
    indices = nbrs.kneighbors(emb, return_distance=False)  # [N, k] or [N, k+1]

    edge_set = set()

    for i in range(n):
        neigh = indices[i].tolist()

        if include_self:
            neigh = [j for j in neigh if j != i]

        for j in neigh[:k]:
            edge_set.add((i, j))

    if mutual:
        edge_set = {(i, j) for (i, j) in edge_set if (j, i) in edge_set}

    # 常见做法：转成无向图
    undirected_edges = set()
    for i, j in edge_set:
        undirected_edges.add((i, j))
        undirected_edges.add((j, i))

    edge_index = torch.tensor(list(undirected_edges), dtype=torch.long).t().contiguous()
    return edge_index

def build_knn_edge_index(x, k=10, metric="cosine"):
    """
    x: torch.Tensor [N, d]
    return: edge_index [2, E]
    """
    x_np = x.detach().cpu().numpy()
    n = x_np.shape[0]

    if n <= 1:
        return torch.empty((2, 0), dtype=torch.long, device=x.device)

    k_eff = min(k, n - 1)
    nbrs = NearestNeighbors(n_neighbors=k_eff + 1, metric=metric)
    nbrs.fit(x_np)
    indices = nbrs.kneighbors(x_np, return_distance=False)

    edges = set()
    for i in range(n):
        neigh = [j for j in indices[i].tolist() if j != i][:k_eff]
        for j in neigh:
            edges.add((i, j))
            edges.add((j, i))  # 无向化

    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long, device=x.device)

    edge_index = torch.tensor(list(edges), dtype=torch.long, device=x.device).t().contiguous()
    return edge_index

In [19]:
# 全局 KNN 建图
def build_global_knn_edge_index(emb, k=10, metric="cosine", device="cpu"):
    """
    emb: np.ndarray or torch.Tensor [N, d]
    return: global edge_index [2, E]
    """
    if isinstance(emb, torch.Tensor):
        emb = emb.detach().cpu().numpy()

    n = emb.shape[0]
    if n <= 1:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    k_eff = min(k, n - 1)
    nbrs = NearestNeighbors(n_neighbors=k_eff + 1, metric=metric)
    nbrs.fit(emb)
    indices = nbrs.kneighbors(emb, return_distance=False)

    edges = set()
    for i in range(n):
        neigh = [j for j in indices[i].tolist() if j != i][:k_eff]
        for j in neigh:
            edges.add((i, j))
            edges.add((j, i))  # 无向化

    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    edge_index = torch.tensor(list(edges), dtype=torch.long, device=device).t().contiguous()
    return edge_index

In [20]:
# 从全局图截取 batch 子图边
def induce_subgraph_edges(global_edge_index, sub_nodes, device="cpu"):
    """
    global_edge_index: [2, E], global node ids
    sub_nodes: 当前 batch 的全局节点编号
    return: local edge_index [2, E_sub]
    """
    if len(sub_nodes) == 0:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    if isinstance(global_edge_index, torch.Tensor):
        global_edge_index = global_edge_index.detach().cpu()

    sub_nodes = list(sub_nodes)
    sub_set = set(sub_nodes)
    global_to_local = {g: i for i, g in enumerate(sub_nodes)}

    src = global_edge_index[0].tolist()
    dst = global_edge_index[1].tolist()

    edges = []
    for s, d in zip(src, dst):
        if s in sub_set and d in sub_set:
            edges.append((global_to_local[s], global_to_local[d]))

    if len(edges) == 0:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    local_edge_index = torch.tensor(edges, dtype=torch.long, device=device).t().contiguous()
    return local_edge_index

#### GAT

In [21]:
class HomoGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=256, heads=4, dropout=0.2):
        super().__init__()
        self.conv1 = GATv2Conv(
            in_dim, hidden_dim, heads=heads,
            concat=True, dropout=dropout, add_self_loops=True
        )
        self.lin1 = nn.Linear(in_dim, hidden_dim * heads)

        self.conv2 = GATv2Conv(
            hidden_dim * heads, out_dim, heads=1,
            concat=False, dropout=dropout, add_self_loops=True
        )
        self.lin2 = nn.Linear(hidden_dim * heads, out_dim)

        self.dropout = dropout

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        x = F.elu(h + self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        h = self.conv2(x, edge_index)
        x = h + self.lin2(x)
        return x

#### HGT架构

In [22]:
class RelTemporalEncoding(nn.Module):
    '''
        Implement the Temporal Encoding (Sinusoid) function.
    '''
    def __init__(self, n_hid, max_len = 240, dropout = 0.2):
        super(RelTemporalEncoding, self).__init__()
        position = torch.arange(0., max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, n_hid, 2) *
                             -(math.log(10000.0) / n_hid))
        emb = nn.Embedding(max_len, n_hid)
        emb.weight.data[:, 0::2] = torch.sin(position * div_term) / math.sqrt(n_hid)
        emb.weight.data[:, 1::2] = torch.cos(position * div_term) / math.sqrt(n_hid)
        emb.requires_grad = False
        self.emb = emb
        self.lin = nn.Linear(n_hid, n_hid)
    def forward(self, x, t):
        return x + self.lin(self.emb(t))

In [23]:
class HGTConv(MessagePassing):
    def __init__(self, in_dim, out_dim, num_types, num_relations, n_heads, dropout = 0.2, use_norm = True, use_RTE = True, **kwargs):
        super(HGTConv, self).__init__(node_dim=0, aggr='add', **kwargs)

        self.in_dim        = in_dim
        self.out_dim       = out_dim
        self.num_types     = num_types
        self.num_relations = num_relations
        self.total_rel     = num_types * num_relations * num_types
        self.n_heads       = n_heads
        self.d_k           = out_dim // n_heads
        self.sqrt_dk       = math.sqrt(self.d_k)
        self.use_norm      = use_norm
        self.use_RTE       = use_RTE
        self.att           = None
        self.res_att        =None
        self.res            =None
        
        
        self.k_linears   = nn.ModuleList()
        self.q_linears   = nn.ModuleList()
        self.v_linears   = nn.ModuleList()
        self.a_linears   = nn.ModuleList()
        self.norms       = nn.ModuleList()
        
        for t in range(num_types):
            self.k_linears.append(nn.Linear(in_dim,   out_dim))
            self.q_linears.append(nn.Linear(in_dim,   out_dim))
            self.v_linears.append(nn.Linear(in_dim,   out_dim))
            self.a_linears.append(nn.Linear(out_dim,  out_dim))
            if use_norm:
                self.norms.append(nn.LayerNorm(out_dim))
        '''
            TODO: make relation_pri smaller, as not all <st, rt, tt> pair exist in meta relation list.
        '''
        self.relation_pri   = nn.Parameter(torch.ones(num_relations, self.n_heads))
        self.relation_att   = nn.Parameter(torch.Tensor(num_relations, n_heads, self.d_k, self.d_k))
        self.relation_msg   = nn.Parameter(torch.Tensor(num_relations, n_heads, self.d_k, self.d_k))
        self.skip           = nn.Parameter(torch.ones(num_types))
        self.drop           = nn.Dropout(dropout)
        
        if self.use_RTE:
            self.emb            = RelTemporalEncoding(in_dim)
        
        glorot(self.relation_att)
        glorot(self.relation_msg)
     
    def _initialize_weights(self):

        for m in self.modules():
            print(m)
            if isinstance(m, nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight, gain=1)

    
    
    def forward(self, node_inp, node_type, edge_index, edge_type, edge_time):
        return self.propagate(edge_index, node_inp=node_inp, node_type=node_type, \
                              edge_type=edge_type, edge_time = edge_time)

    def message(self, edge_index_i, node_inp_i, node_inp_j, node_type_i, node_type_j, edge_type, edge_time):
        '''
            j: source, i: target; <j, i>
        '''
        data_size = edge_index_i.size(0)
        '''
            Create Attention and Message tensor beforehand.
        '''
        self.res_att     = torch.zeros(data_size, self.n_heads).to(node_inp_i.device)
        res_msg     = torch.zeros(data_size, self.n_heads, self.d_k).to(node_inp_i.device)
        
        for source_type in range(self.num_types):
            sb = (node_type_j == int(source_type))
            k_linear = self.k_linears[source_type]
            v_linear = self.v_linears[source_type] 
            for target_type in range(self.num_types):
                tb = (node_type_i == int(target_type)) & sb
                q_linear = self.q_linears[target_type]
                for relation_type in range(self.num_relations):
                    '''
                        idx is all the edges with meta relation <source_type, relation_type, target_type>
                    '''
                    idx = (edge_type == int(relation_type)) & tb
                    if idx.sum() == 0:
                        continue
                    '''
                        Get the corresponding input node representations by idx.
                        Add tempotal encoding to source representation (j)
                    '''
                    target_node_vec = node_inp_i[idx]
                    source_node_vec = node_inp_j[idx]
                    if self.use_RTE:
                        source_node_vec = self.emb(source_node_vec, edge_time[idx])
                    '''
                        Step 1: Heterogeneous Mutual Attention
                    '''
                    q_mat = q_linear(target_node_vec).view(-1, self.n_heads, self.d_k)
                    k_mat = k_linear(source_node_vec).view(-1, self.n_heads, self.d_k)
                    k_mat = torch.bmm(k_mat.transpose(1,0), self.relation_att[relation_type]).transpose(1,0)
                    self.res_att[idx] = (q_mat * k_mat).sum(dim=-1) * self.relation_pri[relation_type] / self.sqrt_dk
                    '''
                        Step 2: Heterogeneous Message Passing
                    '''
                    v_mat = v_linear(source_node_vec).view(-1, self.n_heads, self.d_k)
                    res_msg[idx] = torch.bmm(v_mat.transpose(1,0), self.relation_msg[relation_type]).transpose(1,0)   
        '''
            Softmax based on target node's id (edge_index_i). Store attention value in self.att for later visualization.
        '''
        #self.att = self.res_att
        #attention 
        #self.att =softmax(self.res_att, edge_index_i)
        #embedding in each layer
        #res =res_msg * softmax(self.res_att, edge_index_i).view(-1, self.n_heads, 1)
        res =res_msg * softmax(self.res_att.view(-1, self.n_heads, 1), edge_index_i)
        #self.res_att=self.att
        
        return res.view(-1, self.out_dim)


    def update(self, aggr_out, node_inp, node_type):
        '''
            Step 3: Target-specific Aggregation
            x = W[node_type] * gelu(Agg(x)) + x
        '''
        aggr_out = F.gelu(aggr_out)
        res = torch.zeros(aggr_out.size(0), self.out_dim).to(node_inp.device)
        for target_type in range(self.num_types):
            idx = (node_type == int(target_type))
            if idx.sum() == 0:
                continue
            trans_out = self.drop(self.a_linears[target_type](aggr_out[idx]))
            '''
                Add skip connection with learnable weight self.skip[t_id]
            '''
            alpha = torch.sigmoid(self.skip[target_type])
            if self.use_norm:
                res[idx] = self.norms[target_type](trans_out * alpha + node_inp[idx] * (1 - alpha))
            else:
                res[idx] = trans_out * alpha + node_inp[idx] * (1 - alpha)
        self.res=res
        return res

    def __repr__(self):
        return '{}(in_dim={}, out_dim={}, num_types={}, num_types={})'.format(
            self.__class__.__name__, self.in_dim, self.out_dim,
            self.num_types, self.num_relations)

In [24]:
class DenseHGTConv(MessagePassing):
    def __init__(self, in_dim, out_dim, num_types, num_relations, n_heads, dropout = 0.2, use_norm = True, use_RTE = True, **kwargs):
        super(DenseHGTConv, self).__init__(node_dim=0, aggr='add', **kwargs)

        self.in_dim        = in_dim
        self.out_dim       = out_dim
        self.num_types     = num_types
        self.num_relations = num_relations
        self.total_rel     = num_types * num_relations * num_types
        self.n_heads       = n_heads
        self.d_k           = out_dim // n_heads
        self.sqrt_dk       = math.sqrt(self.d_k)
        self.use_norm      = use_norm
        self.use_RTE       = use_RTE
        self.att           = None
        
        
        self.k_linears   = nn.ModuleList()
        self.q_linears   = nn.ModuleList()
        self.v_linears   = nn.ModuleList()
        self.a_linears   = nn.ModuleList()
        self.norms       = nn.ModuleList()

        
        for t in range(num_types):
            self.k_linears.append(nn.Linear(in_dim,   out_dim))
            self.q_linears.append(nn.Linear(in_dim,   out_dim))
            self.v_linears.append(nn.Linear(in_dim,   out_dim))
            self.a_linears.append(nn.Linear(out_dim,  out_dim))
            if use_norm:
                self.norms.append(nn.LayerNorm(out_dim))
        '''
            TODO: make relation_pri smaller, as not all <st, rt, tt> pair exist in meta relation list.
        '''
        self.relation_pri   = nn.Parameter(torch.ones(num_relations, self.n_heads))
        self.relation_att   = nn.Parameter(torch.Tensor(num_relations, n_heads, self.d_k, self.d_k))
        self.relation_msg   = nn.Parameter(torch.Tensor(num_relations, n_heads, self.d_k, self.d_k))
        self.drop           = nn.Dropout(dropout)
        
        if self.use_RTE:
            self.emb            = RelTemporalEncoding(in_dim)
        
        glorot(self.relation_att)
        glorot(self.relation_msg)
        
        
        self.mid_linear  = nn.Linear(out_dim,  out_dim * 2)
        self.out_linear  = nn.Linear(out_dim * 2,  out_dim)
        self.out_norm    = nn.LayerNorm(out_dim)
        
    def forward(self, node_inp, node_type, edge_index, edge_type, edge_time):
        return self.propagate(edge_index, node_inp=node_inp, node_type=node_type, \
                              edge_type=edge_type, edge_time = edge_time)

    def message(self, edge_index_i, node_inp_i, node_inp_j, node_type_i, node_type_j, edge_type, edge_time):
        '''
            j: source, i: target; <j, i>
        '''
        data_size = edge_index_i.size(0)
        '''
            Create Attention and Message tensor beforehand.
        '''
        res_att     = torch.zeros(data_size, self.n_heads).to(node_inp_i.device)
        res_msg     = torch.zeros(data_size, self.n_heads, self.d_k).to(node_inp_i.device)
        
        for source_type in range(self.num_types):
            sb = (node_type_j == int(source_type))
            k_linear = self.k_linears[source_type]
            v_linear = self.v_linears[source_type] 
            for target_type in range(self.num_types):
                tb = (node_type_i == int(target_type)) & sb
                q_linear = self.q_linears[target_type]
                for relation_type in range(self.num_relations):
                    '''
                        idx is all the edges with meta relation <source_type, relation_type, target_type>
                    '''
                    idx = (edge_type == int(relation_type)) & tb
                    if idx.sum() == 0:
                        continue
                    '''
                        Get the corresponding input node representations by idx.
                        Add tempotal encoding to source representation (j)
                    '''
                    target_node_vec = node_inp_i[idx]
                    source_node_vec = node_inp_j[idx]
                    if self.use_RTE:
                        source_node_vec = self.emb(source_node_vec, edge_time[idx])
                    '''
                        Step 1: Heterogeneous Mutual Attention
                    '''
                    q_mat = q_linear(target_node_vec).view(-1, self.n_heads, self.d_k)
                    k_mat = k_linear(source_node_vec).view(-1, self.n_heads, self.d_k)
                    k_mat = torch.bmm(k_mat.transpose(1,0), self.relation_att[relation_type]).transpose(1,0)
                    res_att[idx] = (q_mat * k_mat).sum(dim=-1) * self.relation_pri[relation_type] / self.sqrt_dk
                    '''
                        Step 2: Heterogeneous Message Passing
                    '''
                    v_mat = v_linear(source_node_vec).view(-1, self.n_heads, self.d_k)
                    res_msg[idx] = torch.bmm(v_mat.transpose(1,0), self.relation_msg[relation_type]).transpose(1,0)   
        '''
            Softmax based on target node's id (edge_index_i). Store attention value in self.att for later visualization.
        '''
        #self.att=res_att
        self.att = softmax(res_att, edge_index_i)
        res = res_msg * self.att.view(-1, self.n_heads, 1)
        del res_att, res_msg
        return res.view(-1, self.out_dim)


    def update(self, aggr_out, node_inp, node_type):
        '''
            Step 3: Target-specific Aggregation
            x = W[node_type] * Agg(x) + x
        '''
        res = torch.zeros(aggr_out.size(0), self.out_dim).to(node_inp.device)
        for target_type in range(self.num_types):
            idx = (node_type == int(target_type))
            if idx.sum() == 0:
                continue
            trans_out = self.drop(self.a_linears[target_type](aggr_out[idx])) + node_inp[idx]
            '''
                Add skip connection with learnable weight self.skip[t_id]
            '''
            if self.use_norm:
                trans_out = self.norms[target_type](trans_out)
                
            '''
                Step 4: Shared Dense Layer
                x = Out_L(gelu(Mid_L(x))) + x
            '''
                
            trans_out     = self.drop(self.out_linear(F.gelu(self.mid_linear(trans_out)))) + trans_out
            res[idx]      = self.out_norm(trans_out)
        return res

    def __repr__(self):
        return '{}(in_dim={}, out_dim={}, num_types={}, num_types={})'.format(
            self.__class__.__name__, self.in_dim, self.out_dim,
            self.num_types, self.num_relations)

In [25]:
class GeneralConv(nn.Module):
    def __init__(self, conv_name, in_hid, out_hid, num_types, num_relations, n_heads, dropout, use_norm = True, use_RTE = True):
        super(GeneralConv, self).__init__()
        self.conv_name = conv_name
        self.res_att = None
        self.res = None
        if self.conv_name == 'hgt':
            self.base_conv = HGTConv(in_hid, out_hid, num_types, num_relations, n_heads, dropout, use_norm, use_RTE)
        elif self.conv_name == 'dense_hgt':
            self.base_conv = DenseHGTConv(in_hid, out_hid, num_types, num_relations, n_heads, dropout, use_norm, use_RTE)
        elif self.conv_name == 'gcn':
            self.base_conv = GCNConv(in_hid, out_hid)
        elif self.conv_name == 'gat':
            self.base_conv = GATConv(in_hid, out_hid // n_heads, heads=n_heads)
    def forward(self, meta_xs, node_type, edge_index, edge_type, edge_time):
        if self.conv_name == 'hgt':
            a=self.base_conv(meta_xs, node_type, edge_index, edge_type, edge_time)
            self.res_att = self.base_conv.res_att
            self.res = self.base_conv.res
            return a
        elif self.conv_name == 'gcn':
            return self.base_conv(meta_xs, edge_index)
        elif self.conv_name == 'gat':
            return self.base_conv(meta_xs, edge_index)
        elif self.conv_name == 'dense_hgt':
            return self.base_conv(meta_xs, node_type, edge_index, edge_type, edge_time)

In [26]:
class GNN_from_raw(nn.Module):
    def __init__(self, in_dim, n_hid, num_types, num_relations, n_heads, n_layers, \
        dropout = 0.2, conv_name = 'hgt', \
        prev_norm = True, last_norm = True, \
        use_RTE = True,\
        AEtype=0\
        ):
        super(GNN_from_raw, self).__init__()
        self.gcs = nn.ModuleList()
        self.num_types = num_types
        self.in_dim    = in_dim
        self.n_hid     = n_hid
        self.adapt_ws  = nn.ModuleList()
        self.drop      = nn.Dropout(dropout)
        self.embedding1 = nn.ModuleList()
        self.embedding2 = nn.ModuleList()
        self.decode1 = nn.ModuleList()
        self.decode2 = nn.ModuleList()
        self.AEtype = AEtype
        self.att =None
        self.conv_name = conv_name
        for ti in range(num_types):
             #self.embedding.append(F.relu(nn.Linear(512,256)(F.relu(nn.Linear(in_dim[ti],512)))))
             self.embedding1.append(nn.Linear(in_dim[ti],512)) #embedding1[0] [2713 x 512] embedding1[1] [24022 x 512]
             self.embedding2.append(nn.Linear(512,256)) #embedding2[0] [512, 256] embedding2[1] [512,256]
        
        if AEtype==1: #embedding autoencoder
           for ti in range(num_types):
                #self.embedding.append(F.relu(nn.Linear(512,256)(F.relu(nn.Linear(in_dim[ti],512)))))
                self.decode1.append(nn.Linear(256,512)) #embedding1[0] [2713 x 512] embedding1[1] [24022 x 512]
                self.decode2.append(nn.Linear(512,in_dim[ti])) #embedding2[0] [512, 256] embedding2[1] [512,256]
        elif AEtype==2:
            for ti in range(num_types):
                #self.embedding.append(F.relu(nn.Linear(512,256)(F.relu(nn.Linear(in_dim[ti],512)))))
                self.decode1.append(nn.Linear(n_hid,512)) #embedding1[0] [2713 x 512] embedding1[1] [24022 x 512]
                self.decode2.append(nn.Linear(512,in_dim[ti])) #embedding2[0] [512, 256] embedding2[1] [512,256]
        
        for t in range(num_types):
            self.adapt_ws.append(nn.Linear(256, n_hid)) #256 could be one additional hyperparameter!!!
        for l in range(n_layers - 1):
            self.gcs.append(GeneralConv(conv_name, n_hid, n_hid, num_types, num_relations, n_heads, dropout, use_norm = prev_norm, use_RTE = use_RTE))
        self.gcs.append(GeneralConv(conv_name, n_hid, n_hid, num_types, num_relations, n_heads, dropout, use_norm = last_norm, use_RTE = use_RTE))
    
    
    def encode(self, x,t_id):
        h1 = F.relu(self.embedding1[t_id](x))
        return F.relu(self.embedding2[t_id](h1))
    
    def decode(self, z,t_id):
        h3 = F.relu(self.decode1[t_id](z))
        return torch.relu(self.decode2[t_id](h3))
        #return torch.relu(self.fc4(z))
    
    def forward(self, node_feature, node_type, edge_time, edge_index, edge_type):
        node_embedding=[] #len = 2 
        for t_id in range(self.num_types):
            node_embedding += list(self.encode(node_feature[t_id],t_id))
        
        node_embedding_stack = torch.stack(node_embedding)
        #print("shape of node_embedding="+str(node_embedding_stack.shape)+"\n")
        res = torch.zeros(node_embedding_stack.size(0), self.n_hid).to(node_feature[0].device)
        
        for t_id in range(self.num_types):
            idx = (node_type == int(t_id)) #0, 1
            if idx.sum() == 0:
                continue
            #res[idx] = torch.tanh(self.adapt_ws[t_id](self.embedding[t_id](node_feature[idx])))
            #print(idx)
            res[idx] = torch.tanh(self.adapt_ws[t_id](node_embedding_stack[idx]))
            #res[idx] = torch.tanh(self.adapt_ws[t_id](self.encode(node_feature[t_id],t_id)))
        
        meta_xs = self.drop(res)
        del res
        for gc in self.gcs:
            meta_xs = gc(meta_xs, node_type, edge_index, edge_type, edge_time)
        if (self.conv_name == 'hgt'):
            self.att = gc.res_att    
        if self.AEtype!=0:
            if self.AEtype==1:#embedding auto-encoder
                 decode_embedding=[]
                 for t_id in range(self.num_types):
                       decode_embedding.append(self.decode(node_embedding_stack[node_type==t_id],t_id)) #0 genematrix 1 cellmatrix
                       #print(decode_embedding[t_id].shape)
                       #print(meta_xs[node_type==t_id].shape)
                 
                 return meta_xs,decode_embedding
            
            elif self.AEtype==2: #HGT embedding auto-encotder 
                 decode_embedding = []
                 for t_id in range(self.num_types):
                       decode_embedding.append(self.decode(meta_xs[node_type==t_id],t_id)) #0 genematrix 1 cellmatrix
                       #print("in model decode_embedding shape tid="+str(t_id))
                       #print(decode_embedding[t_id].shape)
                       #print(meta_xs[node_type==t_id].shape)
                 return meta_xs,decode_embedding
        else:
            return meta_xs  


In [27]:
class GNN(nn.Module):
    def __init__(self, in_dim, n_hid, num_types, num_relations, n_heads, n_layers, dropout = 0.2, conv_name = 'hgt', prev_norm = True, last_norm = True, use_RTE = True):
        super(GNN, self).__init__()
        self.gcs = nn.ModuleList()
        self.num_types = num_types
        self.in_dim    = in_dim
        self.n_hid     = n_hid
        self.adapt_ws  = nn.ModuleList()
        self.drop      = nn.Dropout(dropout)
        self.att =None
        self.emb =None
        self.conv_name = conv_name
        for t in range(num_types):
            self.adapt_ws.append(nn.Linear(in_dim, n_hid))
        for l in range(n_layers - 1):
            self.gcs.append(GeneralConv(conv_name, n_hid, n_hid, num_types, num_relations, n_heads, dropout, use_norm = prev_norm, use_RTE = use_RTE))
        self.gcs.append(GeneralConv(conv_name, n_hid, n_hid, num_types, num_relations, n_heads, dropout, use_norm = last_norm, use_RTE = use_RTE))

    def forward(self, node_feature, node_type, edge_time, edge_index, edge_type):
        res = torch.zeros(node_feature.size(0), self.n_hid).to(node_feature.device)
        for t_id in range(self.num_types):
            idx = (node_type == int(t_id))
            if idx.sum() == 0:
                continue
            res[idx] = torch.tanh(self.adapt_ws[t_id](node_feature[idx]))
        meta_xs = self.drop(res)
        del res
        self.att = {}
        i=0
        self.emb={}
        for gc in self.gcs:
            meta_xs = gc(meta_xs, node_type, edge_index, edge_type, edge_time)
            if (self.conv_name == 'hgt'):
                #self.att = gc.res_att
                self.att[i]=gc.res_att
                #self.emb[i]=gc.res
                i=i+1
                #print(gc.res_att)
                #for p in gc.parameters():
                #    print(p)                
        #self.att = gc.res_att
        self.att = self.att[0]
        return meta_xs  

In [28]:
def compute_kl_reconstruction_loss(gene_z, cell_z, adj_sub):
    """
    gene_z: [n_g, d]
    cell_z: [n_c, d]
    adj_sub: [n_g, n_c]
    """
    decoder = gene_z @ cell_z.T
    loss = F.kl_div(
        decoder.softmax(dim=-1).log(),
        adj_sub.softmax(dim=-1),
        reduction='sum'
    )
    return loss, decoder

In [29]:
def compute_cross_linkpred_loss(node_rep, edge_index):
    """
    node_rep: [n_nodes, d]
    edge_index: [2, n_edges]
    """
    EPS = 1e-15

    # positive edges
    value_pos = (node_rep[edge_index[0]] * node_rep[edge_index[1]]).sum(dim=1)
    pos_loss = -torch.log(torch.sigmoid(value_pos) + EPS).mean()

    # negative edges
    neg_edge_index = negative_sampling(edge_index, num_nodes=node_rep.size(0))
    value_neg = (node_rep[neg_edge_index[0]] * node_rep[neg_edge_index[1]]).sum(dim=1)
    neg_loss = -torch.log(1 - torch.sigmoid(value_neg) + EPS).mean()

    loss = pos_loss + neg_loss
    return loss

In [ ]:
# class GAT_HGT_Wrapper(nn.Module):
#     def __init__(
#         self,
#         hgt_model,
#         in_dim,
#         gat_hidden_dim=128,
#         gat_heads=4,
#         gat_dropout=0.2,
    #     knn_k=10,
    # ):
    #     super().__init__()
    #     self.hgt = hgt_model
    #     self.gene_gat = HomoGAT(
    #         in_dim=in_dim,
    #         hidden_dim=gat_hidden_dim,
    #         out_dim=in_dim,
    #         heads=gat_heads,
    #         dropout=gat_dropout,
    #     )
    #     self.cell_gat = HomoGAT(
    #         in_dim=in_dim,
    #         hidden_dim=gat_hidden_dim,
    #         out_dim=in_dim,
    #         heads=gat_heads,
    #         dropout=gat_dropout,
    #     )
    #     self.knn_k = knn_k

    # def forward(self, node_feature, node_type, edge_time, edge_index, edge_type):
    #     # 这里 node_feature 在 feature_mode='ae' 时是 [N, d]
    #     gene_x = node_feature[node_type == 0]
    #     cell_x = node_feature[node_type == 1]

    #     edge_index_gg = build_knn_edge_index(gene_x, k=self.knn_k)
    #     edge_index_cc = build_knn_edge_index(cell_x, k=self.knn_k)

    #     gene_x = self.gene_gat(gene_x, edge_index_gg)
    #     cell_x = self.cell_gat(cell_x, edge_index_cc)

    #     refined_node_feature = torch.zeros_like(node_feature)
    #     refined_node_feature[node_type == 0] = gene_x
    #     refined_node_feature[node_type == 1] = cell_x

    #     node_rep = self.hgt(
    #         node_feature=refined_node_feature,
    #         node_type=node_type,
    #         edge_time=edge_time,
    #         edge_index=edge_index,
    #         edge_type=edge_type,
    #     )

    #     # 方便后面调试
    #     self.att = getattr(self.hgt, "att", None)
    #     return node_rep
    
class GAT_HGT_Wrapper(nn.Module):
    def __init__(
        self,
        hgt_model,
        in_dim,
        gat_hidden_dim=128,
        gat_heads=4,
        gat_dropout=0.2,
    ):
        super().__init__()
        self.hgt = hgt_model
        self.gene_gat = HomoGAT(
            in_dim=in_dim,
            hidden_dim=gat_hidden_dim,
            out_dim=in_dim,
            heads=gat_heads,
            dropout=gat_dropout,
        )
        self.cell_gat = HomoGAT(
            in_dim=in_dim,
            hidden_dim=gat_hidden_dim,
            out_dim=in_dim,
            heads=gat_heads,
            dropout=gat_dropout,
        )
        self.att = None

    def forward(
        self,
        node_feature,
        node_type,
        edge_time,
        edge_index,
        edge_type,
        edge_index_gg=None,
        edge_index_cc=None,
    ):
        gene_mask = (node_type == 0)
        cell_mask = (node_type == 1)

        gene_x = node_feature[gene_mask]
        cell_x = node_feature[cell_mask]

        if edge_index_gg is None:
            edge_index_gg = torch.empty((2, 0), dtype=torch.long, device=node_feature.device)
        if edge_index_cc is None:
            edge_index_cc = torch.empty((2, 0), dtype=torch.long, device=node_feature.device)

        gene_x = self.gene_gat(gene_x, edge_index_gg)
        cell_x = self.cell_gat(cell_x, edge_index_cc)

        refined_node_feature = torch.zeros_like(node_feature)
        refined_node_feature[gene_mask] = gene_x
        refined_node_feature[cell_mask] = cell_x

        node_rep = self.hgt(
            node_feature=refined_node_feature,
            node_type=node_type,
            edge_time=edge_time,
            edge_index=edge_index,
            edge_type=edge_type,
        )

        self.att = getattr(self.hgt, "att", None)
        return node_rep

In [38]:
# def build_hgt_model(
#     feature_mode: str,
#     gene_embedding_dim: int = None,
#     cell_embedding_dim: int = None,
#     n_hid: int = 128,
#     n_heads: int = 8,
#     n_layers: int = 2,
#     dropout: float = 0.2,
#     AEtype: int = 1,
#     device: str = "cuda" if torch.cuda.is_available() else "cpu",
# ):
#     """
#     Requires:
#         - GNN
#         - GNN_from_raw
#     already defined/imported.
#     """

#     if feature_mode == "ae":
#         assert gene_embedding_dim is not None
#         model = GNN(
#             in_dim=gene_embedding_dim,
#             n_hid=n_hid,
#             num_types=2,
#             num_relations=2,
#             n_heads=n_heads,
#             n_layers=n_layers,
#             dropout=dropout,
#             conv_name='hgt',
#             prev_norm=True,
#             last_norm=True,
#             use_RTE=False,
#         ).to(device)

#     elif feature_mode == "raw":
#         assert gene_embedding_dim is not None and cell_embedding_dim is not None
#         model = GNN_from_raw(
#             in_dim=[gene_embedding_dim, cell_embedding_dim],
#             n_hid=n_hid,
#             num_types=2,
#             num_relations=2,
#             n_heads=n_heads,
#             n_layers=n_layers,
#             dropout=dropout,
#             conv_name='hgt',
#             prev_norm=True,
#             last_norm=True,
#             use_RTE=False,
#             AEtype=AEtype,
#         ).to(device)
#     else:
#         raise ValueError("feature_mode must be 'ae' or 'raw'")

#     return model

def build_hgt_model(
    feature_mode: str,
    gene_embedding_dim: int = None,
    cell_embedding_dim: int = None,
    n_hid: int = 128,
    n_heads: int = 8,
    n_layers: int = 2,
    dropout: float = 0.2,
    AEtype: int = 1,
    use_gat: bool = True,
    gat_hidden_dim: int = 128,
    gat_heads: int = 4,
    gat_dropout: float = 0.2,
    knn_k: int = 10,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    if feature_mode == "ae":
        assert gene_embedding_dim is not None
        hgt_model = GNN(
            in_dim=gene_embedding_dim,
            n_hid=n_hid,
            num_types=2,
            num_relations=2,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
            conv_name='hgt',
            prev_norm=True,
            last_norm=True,
            use_RTE=False,
        ).to(device)

        if use_gat:
            model = GAT_HGT_Wrapper(
                hgt_model=hgt_model,
                in_dim=gene_embedding_dim,
                gat_hidden_dim=gat_hidden_dim,
                gat_heads=gat_heads,
                gat_dropout=gat_dropout,
            ).to(device)
        else:
            model = hgt_model

    elif feature_mode == "raw":
        assert gene_embedding_dim is not None and cell_embedding_dim is not None
        model = GNN_from_raw(
            in_dim=[gene_embedding_dim, cell_embedding_dim],
            n_hid=n_hid,
            num_types=2,
            num_relations=2,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout,
            conv_name='hgt',
            prev_norm=True,
            last_norm=True,
            use_RTE=False,
            AEtype=AEtype,
        ).to(device)
    else:
        raise ValueError("feature_mode must be 'ae' or 'raw'")

    return model

In [32]:
def forward_one_subgraph(
    model,
    batch_data,
    feature_mode: str = "ae",
    loss_type: str = "kl",
):
    """
    batch_data from prepare_subgraph_tensors(...)
    """

    node_feature = batch_data["node_feature"]
    node_type = batch_data["node_type"]
    edge_index = batch_data["edge_index"]
    edge_type = batch_data["edge_type"]
    edge_time = batch_data["edge_time"]
    adj_sub = batch_data["adj_sub"]
    n_g = batch_data["n_g"]

    if feature_mode == "ae":
        node_rep = model(
            node_feature=node_feature,
            node_type=node_type,
            edge_time=edge_time,
            edge_index=edge_index,
            edge_type=edge_type,
            edge_index_gg=batch_data.get("edge_index_gg", None),
            edge_index_cc=batch_data.get("edge_index_cc", None),
        )
    elif feature_mode == "raw":
        # original raw branch returns node_rep, node_decoded_embedding
        node_rep, node_decoded_embedding = model(
            node_feature=node_feature,
            node_type=node_type,
            edge_time=edge_time,
            edge_index=edge_index,
            edge_type=edge_type
        )
    else:
        raise ValueError("feature_mode must be 'ae' or 'raw'")

    gene_z = node_rep[node_type == 0]
    cell_z = node_rep[node_type == 1]

    if loss_type == "kl":
        loss, decoder = compute_kl_reconstruction_loss(gene_z, cell_z, adj_sub)

    elif loss_type == "cross":
        loss = compute_cross_linkpred_loss(node_rep, edge_index)
        decoder = None

    else:
        raise ValueError("loss_type must be 'kl' or 'cross'")

    return {
        "loss": loss,
        "node_rep": node_rep,
        "gene_z": gene_z,
        "cell_z": cell_z,
        "decoder": decoder,
        "attention": getattr(model, "att", None),
    }

In [33]:
# def train_hgt_with_subgraph_sampling(
#     gene_cell: np.ndarray,
#     feature_mode: str = "ae",        # "ae" or "raw"
#     loss_type: str = "kl",           # "kl" or "cross"
#     gene_embedding: np.ndarray = None,
#     cell_embedding: np.ndarray = None,
#     n_hid: int = 128,
#     n_heads: int = 8,
#     n_layers: int = 2,
#     dropout: float = 0.2,
#     lr: float = 1e-3,
#     weight_decay: float = 0.0,
#     max_epochs: int = 100,
#     patience: int = 10,
#     n_batch: int = 50,
#     seed_gene_batch_size: int = 128,
#     sample_depth: int = 2,
#     sample_width_gene_to_cell: int = 64,
#     sample_width_cell_to_gene: int = 64,
#     AEtype: int = 1,
#     device: str = "cuda" if torch.cuda.is_available() else "cpu",
#     verbose: bool = True,
# ):
#     """
#     Main training function.
#     """
#     gene_cell = np.asarray(gene_cell, dtype=np.float32)
#     n_genes, n_cells = gene_cell.shape

#     # build graph
#     graph = build_gene_cell_graph(gene_cell)

#     # prepare jobs
#     jobs = make_subgraph_jobs(
#         graph=graph,
#         n_batch=n_batch,
#         seed_gene_batch_size=seed_gene_batch_size,
#         sample_depth=sample_depth,
#         sample_width_gene_to_cell=sample_width_gene_to_cell,
#         sample_width_cell_to_gene=sample_width_cell_to_gene,
#         seed=0,
#     )

#     # infer input dims
#     if feature_mode == "ae":
#         assert gene_embedding is not None and cell_embedding is not None
#         gene_embedding_dim = gene_embedding.shape[1]
#         cell_embedding_dim = cell_embedding.shape[1]
#         assert gene_embedding_dim == cell_embedding_dim
#     else:
#         gene_embedding_dim = gene_cell.shape[1]   # gene feature dim = n_cells
#         cell_embedding_dim = gene_cell.shape[0]   # cell feature dim = n_genes

#     # build model
#     # model = build_hgt_model(
#     #     feature_mode=feature_mode,
#     #     gene_embedding_dim=gene_embedding_dim,
#     #     cell_embedding_dim=cell_embedding_dim,
#     #     n_hid=n_hid,
#     #     n_heads=n_heads,
#     #     n_layers=n_layers,
#     #     dropout=dropout,
#     #     AEtype=AEtype,
#     #     device=device,
#     # )
#     model = build_hgt_model(
#         feature_mode=feature_mode,
#         gene_embedding_dim=gene_embedding_dim,
#         cell_embedding_dim=cell_embedding_dim,
#         n_hid=n_hid,
#         n_heads=n_heads,
#         n_layers=n_layers,
#         dropout=dropout,
#         AEtype=AEtype,
#         use_gat=True,
#         gat_hidden_dim=128,
#         gat_heads=4,
#         gat_dropout=0.2,
#         knn_k=10,
#         device=device,
#     )

#     optimizer = torch.optim.AdamW(
#         model.parameters(),
#         lr=lr,
#         weight_decay=weight_decay,
#     )

#     best_loss = float("inf")
#     best_state = copy.deepcopy(model.state_dict())
#     wait = 0
#     history = []

#     for epoch in range(max_epochs):
#         model.train()
#         epoch_loss = 0.0
#         print(f"Start epoch {epoch:03d}")
#         for bi, job in enumerate(jobs):
#             if bi % 50 == 0:
#                 print(f"  epoch {epoch:03d} batch {bi}/{len(jobs)}")
#             batch_data = prepare_subgraph_tensors(
#                 gene_cell=gene_cell,
#                 job=job,
#                 feature_mode=feature_mode,
#                 gene_embedding=gene_embedding,
#                 cell_embedding=cell_embedding,
#                 device=device,
#             )

#             out = forward_one_subgraph(
#                 model=model,
#                 batch_data=batch_data,
#                 feature_mode=feature_mode,
#                 loss_type=loss_type,
#             )

#             loss = out["loss"]

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             epoch_loss += float(loss.item())

#         epoch_loss = epoch_loss / max(1, len(jobs))
#         history.append(epoch_loss)

#         if verbose and (epoch % 5 == 0 or epoch == max_epochs - 1):
#             print(f"Epoch {epoch:03d} | avg_loss={epoch_loss:.6f}")

#         if epoch_loss < best_loss - 1e-6:
#             best_loss = epoch_loss
#             best_state = copy.deepcopy(model.state_dict())
#             wait = 0
#         else:
#             wait += 1
#             if wait >= patience:
#                 if verbose:
#                     print(f"Early stopping at epoch {epoch:03d}, best_loss={best_loss:.6f}")
#                 break

#     model.load_state_dict(best_state)
#     return {
#         "model": model,
#         "graph": graph,
#         "jobs": jobs,
#         "history": history,
#         "best_loss": best_loss,
#     }

In [34]:
def train_hgt_with_subgraph_sampling(
    gene_cell: np.ndarray,
    feature_mode: str = "ae",        # "ae" or "raw"
    loss_type: str = "kl",           # "kl" or "cross"
    gene_embedding: np.ndarray = None,
    cell_embedding: np.ndarray = None,
    n_hid: int = 128,
    n_heads: int = 8,
    n_layers: int = 2,
    dropout: float = 0.2,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    max_epochs: int = 100,
    patience: int = 10,
    n_batch: int = 50,
    seed_gene_batch_size: int = 128,
    sample_depth: int = 2,
    sample_width_gene_to_cell: int = 64,
    sample_width_cell_to_gene: int = 64,
    AEtype: int = 1,

    # 新增：全局 KNN 图参数
    use_gat: bool = True,
    knn_k: int = 10,
    knn_metric: str = "cosine", # "euclidean" or "cosine"
    gat_hidden_dim: int = 128,
    gat_heads: int = 4,
    gat_dropout: float = 0.2,

    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    verbose: bool = True,
):
    """
    Main training function.
    """
    gene_cell = np.asarray(gene_cell, dtype=np.float32)
    n_genes, n_cells = gene_cell.shape

    # build graph
    graph = build_gene_cell_graph(gene_cell)

    # prepare jobs
    jobs = make_subgraph_jobs(
        graph=graph,
        n_batch=n_batch,
        seed_gene_batch_size=seed_gene_batch_size,
        sample_depth=sample_depth,
        sample_width_gene_to_cell=sample_width_gene_to_cell,
        sample_width_cell_to_gene=sample_width_cell_to_gene,
        seed=0,
    )

    # infer input dims
    if feature_mode == "ae":
        assert gene_embedding is not None and cell_embedding is not None
        gene_embedding_dim = gene_embedding.shape[1]
        cell_embedding_dim = cell_embedding.shape[1]
        assert gene_embedding_dim == cell_embedding_dim
    else:
        gene_embedding_dim = gene_cell.shape[1]   # gene feature dim = n_cells
        cell_embedding_dim = gene_cell.shape[0]   # cell feature dim = n_genes

    # build model
    model = build_hgt_model(
        feature_mode=feature_mode,
        gene_embedding_dim=gene_embedding_dim,
        cell_embedding_dim=cell_embedding_dim,
        n_hid=n_hid,
        n_heads=n_heads,
        n_layers=n_layers,
        dropout=dropout,
        AEtype=AEtype,
        use_gat=use_gat,
        gat_hidden_dim=gat_hidden_dim,
        gat_heads=gat_heads,
        gat_dropout=gat_dropout,
        knn_k=knn_k,
        device=device,
    )

    # 关键新增：训练前全局构一次 gg / cc 图
    global_edge_index_gg = None
    global_edge_index_cc = None

    if feature_mode == "ae" and use_gat:
        if verbose:
            print("Building global gene-gene KNN graph...", flush=True)
        global_edge_index_gg = build_global_knn_edge_index(
            gene_embedding,
            k=knn_k,
            metric=knn_metric,
            device="cpu",
        )

        if verbose:
            print("Building global cell-cell KNN graph...", flush=True)
        global_edge_index_cc = build_global_knn_edge_index(
            cell_embedding,
            k=knn_k,
            metric=knn_metric,
            device="cpu",
        )

        if verbose:
            print(
                f"Global gg edges: {global_edge_index_gg.size(1)}, "
                f"Global cc edges: {global_edge_index_cc.size(1)}",
                flush=True
            )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0

        if verbose:
            print(f"Start epoch {epoch:03d}", flush=True)

        for bi, job in enumerate(jobs):
            if verbose and bi % 10 == 0:
                print(f"  epoch {epoch:03d} batch {bi}/{len(jobs)}", flush=True)

            batch_data = prepare_subgraph_tensors(
                gene_cell=gene_cell,
                job=job,
                feature_mode=feature_mode,
                gene_embedding=gene_embedding,
                cell_embedding=cell_embedding,
                device=device,
            )

            # 关键新增：从全局 gg / cc 图里截取当前 batch 的局部边
            if feature_mode == "ae" and use_gat:
                sub_genes = job["sub_genes"]
                sub_cells = job["sub_cells"]

                edge_index_gg = induce_subgraph_edges(
                    global_edge_index_gg,
                    sub_genes,
                    device=device,
                )
                edge_index_cc = induce_subgraph_edges(
                    global_edge_index_cc,
                    sub_cells,
                    device=device,
                )

                batch_data["edge_index_gg"] = edge_index_gg
                batch_data["edge_index_cc"] = edge_index_cc

            out = forward_one_subgraph(
                model=model,
                batch_data=batch_data,
                feature_mode=feature_mode,
                loss_type=loss_type,
            )

            loss = out["loss"]

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.item())

        epoch_loss = epoch_loss / max(1, len(jobs))
        history.append(epoch_loss)

        if verbose:
            print(f"Epoch {epoch:03d} | avg_loss={epoch_loss:.6f}", flush=True)

        if epoch_loss < best_loss - 1e-6:
            best_loss = epoch_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch:03d}, best_loss={best_loss:.6f}", flush=True)
                break

    model.load_state_dict(best_state)
    return {
        "model": model,
        "graph": graph,
        "jobs": jobs,
        "history": history,
        "best_loss": best_loss,
        "global_edge_index_gg": global_edge_index_gg,
        "global_edge_index_cc": global_edge_index_cc,
    }

In [42]:
@torch.no_grad()
def infer_hgt_embeddings_with_global_graph(
    model,
    gene_cell,
    gene_embedding,
    cell_embedding,
    global_edge_index_gg,
    global_edge_index_cc,
    gene_block_size=1024,
    device="cuda" if torch.cuda.is_available() else "cpu",
):
    model.eval()

    gene_cell = np.asarray(gene_cell, dtype=np.float32)
    n_genes, n_cells = gene_cell.shape
    all_cells = list(range(n_cells))

    refined_gene_blocks = []
    final_cell_embedding = None

    for start in range(0, n_genes, gene_block_size):
        end = min(start + gene_block_size, n_genes)
        sub_genes = list(range(start, end))
        sub_cells = all_cells

        adj_sub = gene_cell[np.ix_(sub_genes, sub_cells)]
        g_local, c_local = np.nonzero(adj_sub)
        edges_gc = list(zip(g_local.tolist(), c_local.tolist()))

        job = {
            "sub_genes": sub_genes,
            "sub_cells": sub_cells,
            "edges_gc": edges_gc,
        }

        batch_data = prepare_subgraph_tensors(
            gene_cell=gene_cell,
            job=job,
            feature_mode="ae",
            gene_embedding=gene_embedding,
            cell_embedding=cell_embedding,
            device=device,
        )

        # 加局部 gg / cc 边
        batch_data["edge_index_gg"] = induce_subgraph_edges(
            global_edge_index_gg, sub_genes, device=device
        )
        batch_data["edge_index_cc"] = induce_subgraph_edges(
            global_edge_index_cc, sub_cells, device=device
        )

        node_rep = model(
            node_feature=batch_data["node_feature"],
            node_type=batch_data["node_type"],
            edge_time=batch_data["edge_time"],
            edge_index=batch_data["edge_index"],
            edge_type=batch_data["edge_type"],
            edge_index_gg=batch_data["edge_index_gg"],
            edge_index_cc=batch_data["edge_index_cc"],
        )

        node_type = batch_data["node_type"]
        gene_z = node_rep[node_type == 0].detach().cpu().numpy()
        cell_z = node_rep[node_type == 1].detach().cpu().numpy()

        refined_gene_blocks.append(gene_z)
        final_cell_embedding = cell_z

    refined_gene_embedding = np.vstack(refined_gene_blocks)

    return {
        "refined_gene_embedding": refined_gene_embedding,
        "refined_cell_embedding": final_cell_embedding,
    }

# @torch.no_grad()
# def infer_hgt_embeddings(
#     model,
#     gene_cell: np.ndarray,
#     feature_mode: str = "ae",
#     gene_embedding: np.ndarray = None,
#     cell_embedding: np.ndarray = None,
#     gene_block_size: int = 1024,
#     device: str = "cuda" if torch.cuda.is_available() else "cpu",
# ):
#     """
#     Infer refined embeddings block-wise over genes.
#     Returns:
#         refined_gene_embedding
#         refined_cell_embedding
#         attention_list
#     """
#     model.eval()

#     gene_cell = np.asarray(gene_cell, dtype=np.float32)
#     n_genes, n_cells = gene_cell.shape

#     refined_gene_blocks = []
#     final_cell_embedding = None
#     attention_blocks = []

#     all_cells = list(range(n_cells))

#     for start in range(0, n_genes, gene_block_size):
#         end = min(start + gene_block_size, n_genes)
#         sub_genes = list(range(start, end))
#         sub_cells = all_cells

#         # edges from submatrix
#         adj_sub = gene_cell[np.ix_(sub_genes, sub_cells)]
#         g_local, c_local = np.nonzero(adj_sub)
#         edges_gc = list(zip(g_local.tolist(), c_local.tolist()))

#         job = {
#             "sub_genes": sub_genes,
#             "sub_cells": sub_cells,
#             "edges_gc": edges_gc,
#         }

#         batch_data = prepare_subgraph_tensors(
#             gene_cell=gene_cell,
#             job=job,
#             feature_mode=feature_mode,
#             gene_embedding=gene_embedding,
#             cell_embedding=cell_embedding,
#             device=device,
#         )

#         if feature_mode == "ae":
#             node_rep = model(
#                 node_feature=batch_data["node_feature"],
#                 node_type=batch_data["node_type"],
#                 edge_time=batch_data["edge_time"],
#                 edge_index=batch_data["edge_index"],
#                 edge_type=batch_data["edge_type"],
#             )
#         else:
#             node_rep, _ = model(
#                 node_feature=batch_data["node_feature"],
#                 node_type=batch_data["node_type"],
#                 edge_time=batch_data["edge_time"],
#                 edge_index=batch_data["edge_index"],
#                 edge_type=batch_data["edge_type"],
#             )

#         node_type = batch_data["node_type"]
#         gene_z = node_rep[node_type == 0].detach().cpu().numpy()
#         cell_z = node_rep[node_type == 1].detach().cpu().numpy()

#         refined_gene_blocks.append(gene_z)
#         final_cell_embedding = cell_z   # all cells every block, keep last
#         attention_blocks.append(getattr(model, "att", None))

#     refined_gene_embedding = np.vstack(refined_gene_blocks)

#     return {
#         "refined_gene_embedding": refined_gene_embedding,
#         "refined_cell_embedding": final_cell_embedding,
#         "attention_blocks": attention_blocks,
#     }

#### HGT训练

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = build_hgt_model(
    feature_mode="ae",
    gene_embedding_dim=256,
    cell_embedding_dim=256,
    n_hid=128,
    n_heads=8,
    n_layers=2,
    dropout=0.2,
    AEtype=1,
    use_gat=True,
    gat_hidden_dim=128,
    gat_heads=4,
    gat_dropout=0.2,
    knn_k=10,
    device=device,
)

print(model)

In [ ]:
train_result = train_hgt_with_subgraph_sampling(
    gene_cell=gene_cell,
    feature_mode="ae",
    loss_type="kl",
    gene_embedding=gene_embedding,
    cell_embedding=cell_embedding,
    n_hid=128,
    n_heads=8,
    n_layers=2,
    dropout=0.2,
    lr=1e-3,
    weight_decay=0.0,
    max_epochs=20,
    patience=10,
    n_batch=50, # number of subgraph batches per epoch 如果把seed_gene_batch_size、sample_width_gene_to_cell、sample_width_cell_to_gene三个参数调大了，建议反而把它调小，因为大 batch 子图已经更有信息量，不需要那么多小子图重复训练，每个 epoch 的 Python/采样开销会明显下降
    seed_gene_batch_size=128,   # Size of the batch of genes to sample in each iteration 每个 batch 覆盖更多 seed genes 单步更“值钱”
    sample_depth=2,
    sample_width_gene_to_cell=64,   # 每个 seed gene 连接的 cell 数量，过大过小都不好，过大噪声多，过小信息不足 每个 seed gene 扩展到更多 cells gene-cell 关系学得更充分
    sample_width_cell_to_gene=64,   # 每个 seed cell 连接的 gene 数量，过大过小都不好，过大噪声多，过小信息不足 每个 seed cell 扩展到更多 genes gene-cell 关系学得更充分
    use_gat=True,
    knn_k=10,
    knn_metric="cosine",
    gat_hidden_dim=128,
    gat_heads=4,
    gat_dropout=0.2,
    verbose=True,
)

Building global gene-gene KNN graph...
Building global cell-cell KNN graph...
Global gg edges: 31258, Global cc edges: 13778
Start epoch 000
  epoch 000 batch 0/50
  epoch 000 batch 10/50
  epoch 000 batch 20/50
  epoch 000 batch 30/50
  epoch 000 batch 40/50
Epoch 000 | avg_loss=10227.527529
Start epoch 001
  epoch 001 batch 0/50
  epoch 001 batch 10/50
  epoch 001 batch 20/50
  epoch 001 batch 30/50
  epoch 001 batch 40/50
Epoch 001 | avg_loss=6227.635029
Start epoch 002
  epoch 002 batch 0/50
  epoch 002 batch 10/50
  epoch 002 batch 20/50
  epoch 002 batch 30/50
  epoch 002 batch 40/50
Epoch 002 | avg_loss=6039.646152
Start epoch 003
  epoch 003 batch 0/50
  epoch 003 batch 10/50
  epoch 003 batch 20/50
  epoch 003 batch 30/50
  epoch 003 batch 40/50
Epoch 003 | avg_loss=5929.525215
Start epoch 004
  epoch 004 batch 0/50
  epoch 004 batch 10/50
  epoch 004 batch 20/50
  epoch 004 batch 30/50
  epoch 004 batch 40/50
Epoch 004 | avg_loss=5874.646797
Start epoch 005
  epoch 005 batch 

#### 计算gene-cell score 矩阵

In [44]:
#### 推理
infer_result = infer_hgt_embeddings_with_global_graph(
    model=train_result["model"],
    gene_cell=gene_cell,
    gene_embedding=gene_embedding,
    cell_embedding=cell_embedding,
    global_edge_index_gg=train_result["global_edge_index_gg"],
    global_edge_index_cc=train_result["global_edge_index_cc"],
    gene_block_size=1024
)

refined_gene_embedding = infer_result["refined_gene_embedding"]
refined_cell_embedding = infer_result["refined_cell_embedding"]

In [59]:
#### 计算重构矩阵 就是gene-cell score矩阵
S_gc = refined_gene_embedding @ refined_cell_embedding.T

# cell-wise z-score
S_gc_z = (S_gc - S_gc.mean(axis=0, keepdims=True)) / (
    S_gc.std(axis=0, keepdims=True) + 1e-6
)
print(S_gc_z.shape)   # [n_genes, n_cells]

# softmax
S_shift = S_gc - S_gc.max(axis=0, keepdims=True)
S_gc_soft = np.exp(S_shift) / np.exp(S_shift).sum(axis=0, keepdims=True)
print(S_gc_soft.shape)   # [n_genes, n_cells]

(2000, 1000)
(2000, 1000)


#### 基于某个Pathway计算得分
- 已有pathway的gene index

In [60]:
seed_idx = [1, 10, 23, 100]

In [ ]:
# seed_idx = pathway gene indices 用Z-score矩阵里对应行的平均值作为这个pathway的score
pathway_score_Zscore = S_gc_z[seed_idx].mean(axis=0)

# Softmax
pathway_score_Softmax = S_gc_soft[seed_idx].sum(axis=0)